In [1]:
city = "cologne"
date = "20250718"

bicycle_pt = "walking"
path = f"../data/{date}/mcr5_results"

selected_path = f"{path}/{city}/{bicycle_pt}/errors.pkl"

In [2]:
import mcr_py.utils.storage

errors = mcr_py.utils.storage.read_any_dict(selected_path)

In [3]:
errors[0]

{'h3_cell': '881fa18883fffff',
 'osm_node_id': 1611111990,
 'start_time': '08:00:00',
 'max_transfers': 0,
 'output_path': '/home/ppeter/repo/mcr-py/data/20250718/mcr5_results/cologne/walking/881fa18883fffff.feather',
 'error': "ValueError('Node 1611111990 not found in graph cache - aborting walking step. Current number of Bags is 1')",
 'logs': "2025-08-10 12:34:22,553 - mcr5-881fa18883fffff - DEBUG - Starting MCR with config: {'disable_paths': True, 'path_manager': None, 'output_format': <OutputFormat.DF_FEATHER: 'df_feather'>, 'logger': <Logger mcr5-881fa18883fffff (DEBUG)>, 'timer': <mcr_py.utils.logger.Timer object at 0x7f35923cd750>, 'enable_limit': True, 'initial_steps': [[WalkingStep]], 'repeating_steps': []}\n2025-08-10 12:34:22,553 - mcr5-881fa18883fffff - DEBUG - Starting MCR with config: {'disable_paths': True, 'path_manager': None, 'output_format': <OutputFormat.DF_FEATHER: 'df_feather'>, 'logger': <Logger mcr5-881fa18883fffff (DEBUG)>, 'timer': <mcr_py.utils.logger.Timer 

In [4]:
unknowns_jul = [int(error["error"].split(" ")[1]) for error in errors]

In [5]:
unknowns_aug = [int(error["error"].split(" ")[1]) for error in errors]

In [ ]:
for unknown in unknowns_jul:
    if unknown not in unknowns_aug:
        print(unknown)
    else:
        print("found", unknown)
print("Only aug")
for unknown in unknowns_aug:
    if unknown not in unknowns_jul:
        print(unknown)

found 1611111990
found 8422874017
found 1079741251
found 353246208
found 3281228299
found 8491601318
found 4718689028
found 2399448512
found 2399257273
found 522644748
found 2266210668
found 1611111401
Only aug


In [7]:
import polars as pl
import polars_h3 as plh3
import polars_st as st

h3mapping = pl.read_parquet("../data/20250808/cache/koeln_walking_h3mapping.parquet").lazy()
walking_nodes = pl.read_parquet("../data/20250808/cache/koeln_walking_nodes.parquet").lazy()
walking_edges = pl.read_parquet("../data/20250808/cache/koeln_walking_edges.parquet").lazy()

In [8]:
h3mapping_filt = h3mapping.with_columns(
    st.from_coords(pl.concat_arr(pl.col("lon"), pl.col("lat")))
    .st.set_srid(4326)
    .st.to_srid(4839)
    .st.distance(
        st.from_coords(pl.concat_arr(pl.col("center_lon"), pl.col("center_lat")))
        .st.set_srid(4326)
        .st.to_srid(4839)
    )
    .cast(pl.Float64)
    .alias("distance_to_cell_id_center"),
).select(
    "osm_node_id",
    "h3_cell_id",
    "distance_to_cell_id_center",
    "dist",
)

In [ ]:
test_pl_query = (
    walking_nodes.with_columns(
        plh3.latlng_to_cell(
            pl.col("lat"), pl.col("long"), resolution=8, return_dtype=pl.String
        ).alias("h3_cell"),
        st.point(pl.concat_arr("long", "lat")).st.set_srid(4326).alias("point_lnglat"),
    )
    .with_columns(
        st.point(
            pl.concat_arr(
                plh3.cell_to_lng(pl.col("h3_cell")), plh3.cell_to_lat(pl.col("h3_cell"))
            )
        )
        .st.set_srid(4326)
        .alias("h3_cell_lnglat")
    )
    .with_columns(
        st.to_srid("point_lnglat", srid=4839)
        .st.distance(st.to_srid("h3_cell_lnglat", srid=4839))
        .alias("distance_to_cell"),
        pl.concat_arr(pl.col("lat"), pl.col("long")).alias("point_latlng"),
        plh3.cell_to_latlng(pl.col("h3_cell")).alias("h3_cell_latlng"),
    )
    .group_by("h3_cell")
    .agg(pl.all().sort_by("distance_to_cell").first())
    .select("osm_id", "h3_cell", "point_latlng", "h3_cell_latlng", "distance_to_cell")
)

In [15]:
test_pl_query.collect()

osm_id,h3_cell,point_latlng,h3_cell_latlng,distance_to_cell
u64,str,"array[f64, 2]",list[f64],f64
268782036,"""881fa56de1fffff""","[50.907015, 6.6719347]","[50.90611, 6.665576]",458.009565
1741653856,"""881fa18a61fffff""","[50.926674, 6.9840563]","[50.926593, 6.983933]",12.475761
1689384712,"""881fa18351fffff""","[51.041416, 7.0669111]","[51.041553, 7.066909]",15.210024
2343484777,"""881fa1d715fffff""","[50.858375, 6.7484279]","[50.858436, 6.748497]",8.362377
8918703747,"""881fa564cbfffff""","[51.108703, 6.8443755]","[51.107858, 6.844848]",99.53461
…,…,…,…,…
700748725,"""881fa18c69fffff""","[50.911075, 7.2387226]","[50.91119, 7.238376]",27.495503
1497301139,"""881fa11239fffff""","[50.815643, 7.1689017]","[50.815308, 7.16903]",38.267079
3854902111,"""881fa1819bfffff""","[50.975085, 7.2059675]","[50.975157, 7.206269]",22.624735


In [26]:
result_diff = (
    h3mapping_filt.join(test_pl_query, left_on="h3_cell_id", right_on="h3_cell")
    .filter(
        pl.col("distance_to_cell_id_center").round(0) > pl.col("distance_to_cell").round(0)
    )
    .collect()
)

In [20]:
filtered_cells = h3mapping.filter(pl.col("osm_node_id").is_in(unknowns_aug)).join(
    walking_nodes, left_on="osm_node_id", right_on="osm_id"
)

In [28]:
hex_map = plh3.graphing.plot_hex_outlines(result_diff, "h3_cell_id")
hex_map